# TD6 : RAG avec RDF/SPARQL et un LLM local

On va construire un chatbot qui répond à des questions sur notre KB musicale.

Le principe :
1. L'utilisateur pose une question en français
2. Le LLM (Gemma 2B) traduit la question en SPARQL
3. On exécute la requête sur notre KB avec rdflib
4. Si la requête échoue, le LLM la corrige automatiquement
5. On affiche la réponse



### Configuration

In [1]:
import re
import json
import requests
from rdflib import Graph
from typing import List, Tuple

CHEMIN      = "C:/Users/sandy/OneDrive/Desktop/Cours A4/Web Datamining/TD4_Project/"
TTL_FILE    = CHEMIN + "aligned_kb.ttl"   # KB alignee (plus legere et mieux structuree)
OLLAMA_URL  = "http://localhost:11434/api/generate"
MODEL = "llama3.2:3b"

MAX_PREDICATES = 80
MAX_CLASSES    = 40
SAMPLE_TRIPLES = 20

print(" OK")
print(f"KB : {TTL_FILE}")
print(f"LLM : {MODEL} via {OLLAMA_URL}")

 OK
KB : C:/Users/sandy/OneDrive/Desktop/Cours A4/Web Datamining/TD4_Project/aligned_kb.ttl
LLM : llama3.2:3b via http://localhost:11434/api/generate


### 0. Fonction utilitaire : appel au LLM local

In [2]:
def ask_llm(prompt: str, model: str = MODEL) -> str:
    """Envoie un prompt a Gemma via Ollama et retourne la reponse."""
    payload = {"model": model, "prompt": prompt, "stream": False}
    try:
        r = requests.post(OLLAMA_URL, json=payload, timeout=120)
        if r.status_code != 200:
            raise RuntimeError(f"Ollama erreur {r.status_code}: {r.text}")
        return r.json().get("response", "")
    except requests.exceptions.ConnectionError:
        raise RuntimeError("Ollama ne repond pas. Lance 'ollama serve' dans un terminal.")

# Test de connexion
reponse_test = ask_llm("Reponds en une phrase : qu'est-ce que SPARQL ?")
print("Test LLM OK :")
print(reponse_test[:200])

Test LLM OK :
SPARQL (SPARQL Protocol and RDF Query Language) est un langage de requête utilisé pour accéder et manipuler des données RDF (Resource Description Framework), qui sont des formats de données de metadat


### 1. Chargement du graphe RDF

In [3]:
g = Graph()
g.parse(TTL_FILE, format="turtle")
print(f"Graphe charge : {len(g)} triplets")

Graphe charge : 1918 triplets


### 2. Construction du schema summary

Le schema summary est un résumé compact de notre KB qu'on donne au LLM
pour qu'il sache quels prédicats et classes existent.
Sans ce résumé, le LLM inventerait des prédicats qui n'existent pas.

In [4]:
def get_prefixes(g: Graph) -> str:
    defaults = {
        "rdf":  "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
        "rdfs": "http://www.w3.org/2000/01/rdf-schema#",
        "xsd":  "http://www.w3.org/2001/XMLSchema#",
        "owl":  "http://www.w3.org/2002/07/owl#",
        "ex":   "http://musickg.example.org/resource/",
        "exo":  "http://musickg.example.org/ontology/",
    }
    ns_map = {p: str(ns) for p, ns in g.namespace_manager.namespaces()}
    for k, v in defaults.items():
        ns_map.setdefault(k, v)
    return "\n".join(sorted(f"PREFIX {p}: <{ns}>" for p, ns in ns_map.items()))

def get_predicates(g: Graph, limit=MAX_PREDICATES) -> List[str]:
    q = f"SELECT DISTINCT ?p WHERE {{ ?s ?p ?o . }} LIMIT {limit}"
    return [str(r.p) for r in g.query(q)]

def get_classes(g: Graph, limit=MAX_CLASSES) -> List[str]:
    q = f"SELECT DISTINCT ?cls WHERE {{ ?s a ?cls . }} LIMIT {limit}"
    return [str(r.cls) for r in g.query(q)]

def get_samples(g: Graph, limit=SAMPLE_TRIPLES) -> List[Tuple]:
    q = f"SELECT ?s ?p ?o WHERE {{ ?s ?p ?o . FILTER(isURI(?o)) }} LIMIT {limit}"
    return [(str(r.s), str(r.p), str(r.o)) for r in g.query(q)]

def build_schema(g: Graph) -> str:
    prefixes = get_prefixes(g)
    preds    = get_predicates(g)
    classes  = get_classes(g)
    samples  = get_samples(g)

    # On ajoute des exemples specifiques a notre KB musicale
    exemples_specifiques = """
# Exemples de requetes valides sur cette KB :
# - Artistes : ex:The_Beatles, ex:Daft_Punk, ex:Radiohead
# - Genres   : ex:Genre_rock, ex:Genre_electronic, ex:Genre_hip_hop
# - Labels   : ex:Label_Parlophone, ex:Label_EMI
# - Predicats cles : exo:hasGenre, exo:signedTo, exo:influencedBy, exo:hasMember, exo:releasedAlbum
# - Pour trouver un artiste par nom : ?artiste rdfs:label "The Beatles"@en
"""

    return f"""{prefixes}

# Predicats disponibles (jusqu'a {MAX_PREDICATES})
{chr(10).join(f'- {p}' for p in preds)}

# Classes disponibles (jusqu'a {MAX_CLASSES})
{chr(10).join(f'- {c}' for c in classes)}

# Exemples de triplets
{chr(10).join(f'- {s} {p} {o}' for s, p, o in samples)}
{exemples_specifiques}""".strip()

schema = build_schema(g)
print("Schema summary construit.")
print(f"Longueur : {len(schema)} caracteres")
print()
print(schema[:800])
print("...")

Schema summary construit.
Longueur : 6457 caracteres

PREFIX brick: <https://brickschema.org/schema/Brick#>
PREFIX csvw: <http://www.w3.org/ns/csvw#>
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX dc: <http://purl.org/dc/elements/1.1/>
PREFIX dcam: <http://purl.org/dc/dcam/>
PREFIX dcat: <http://www.w3.org/ns/dcat#>
PREFIX dcmitype: <http://purl.org/dc/dcmitype/>
PREFIX dcterms: <http://purl.org/dc/terms/>
PREFIX doap: <http://usefulinc.com/ns/doap#>
PREFIX ex: <http://musickg.example.org/resource/>
PREFIX exo: <http://musickg.example.org/ontology/>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
PREFIX geo: <http://www.opengis.net/ont/geosparql#>
PREFIX odrl: <http://www.w3.org/ns/odrl/2/>
PREFIX org: <http://www.w3.org/ns/org#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX prof: <http://www.w3.org/ns/dx/prof/>
PREFIX prov: <http://www.w3
...


### 3. Baseline : question sans RAG

On pose une question directement au LLM sans lui donner notre KB.
Le LLM va répondre depuis sa mémoire 

In [5]:
def reponse_baseline(question: str) -> str:
    """Repond a une question sans utiliser la KB."""
    prompt = f"Reponds a cette question en francais : {question}"
    return ask_llm(prompt)

# Test baseline
question_test = "Quel label discographique a signe Radiohead ?"
print(f"Question : {question_test}")
print()
print("Reponse baseline (sans KB) :")
print(reponse_baseline(question_test))

Question : Quel label discographique a signe Radiohead ?

Reponse baseline (sans KB) :
Le label discographique qui a signé Radiohead est le suivant :

Epic Records


### 4. Generation de SPARQL depuis le langage naturel

In [6]:
INSTRUCTIONS_SPARQL = """
Tu es un generateur de requetes SPARQL. Convertis la QUESTION en une requete SPARQL 1.1 SELECT valide
pour le graphe RDF decrit dans le SCHEMA.

Regles strictes :
- Utilise UNIQUEMENT les prefixes et predicats visibles dans le SCHEMA.
- Pour chercher un artiste par nom, utilise : ?artiste rdfs:label "Nom"@en
- Retourne SEULEMENT la requete SPARQL dans un bloc de code ```sparql
- Pas d'explications, pas de texte en dehors du bloc de code.
- La requete doit commencer par SELECT et inclure les prefixes necessaires.
}
"""

CODE_RE = re.compile(r"```(?:sparql)?\s*(.*?)```", re.IGNORECASE | re.DOTALL)

def extraire_sparql(texte: str) -> str:
    # Supprime les blocs de code markdown
    texte = texte.strip()
    # Cas 1 : bloc ```sparql ... ```
    m = re.search(r"```(?:sparql)?\s*(SELECT.*?)```", texte, re.IGNORECASE | re.DOTALL)
    if m:
        return m.group(1).strip()
    # Cas 2 : le texte commence par ```sparql
    texte = re.sub(r"^```sparql\s*", "", texte, flags=re.IGNORECASE)
    texte = re.sub(r"```$", "", texte).strip()
    # Cas 3 : cherche SELECT directement
    m = re.search(r"(SELECT\s+.*)", texte, re.IGNORECASE | re.DOTALL)
    if m:
        return m.group(1).strip()
    return texte

def generer_sparql(question: str, schema: str) -> str:
    prompt = f"{INSTRUCTIONS_SPARQL}\n\nSCHEMA:\n{schema}\n\nQUESTION:\n{question}"
    raw = ask_llm(prompt)
    return extraire_sparql(raw)

# Test
sparql_test = generer_sparql(question_test, schema)
print("SPARQL genere :")
print(sparql_test)

SPARQL genere :
SELECT ?label 
WHERE {
  ?radiohead exo:signedTo ?label.
}


### 5. Execution SPARQL + mecanisme de reparation automatique

Si la requete generee echoue, on envoie l'erreur au LLM
et on lui demande de corriger. C'est le mecanisme de self-repair.

In [10]:
import unicodedata

def normaliser(texte: str) -> str:
    """Supprime les accents d'un texte."""
    return ''.join(
        c for c in unicodedata.normalize('NFD', texte)
        if unicodedata.category(c) != 'Mn'
    )

def executer_sparql(g: Graph, query: str):
    res   = g.query(query)
    vars_ = [str(v) for v in res.vars]
    rows  = [tuple(str(c) for c in r) for r in res]
    return vars_, rows

def reparer_sparql(schema: str, question: str, mauvaise_requete: str, erreur: str) -> str:
    prompt = f"""{INSTRUCTIONS_REPARATION}

SCHEMA:\n{schema}

QUESTION ORIGINALE:\n{question}

MAUVAISE REQUETE:\n{mauvaise_requete}

MESSAGE D'ERREUR:\n{erreur}

Retourne uniquement la requete corrigee."""
    raw = ask_llm(prompt)
    return extraire_sparql(raw)

def repondre_avec_rag(g: Graph, schema: str, question: str) -> dict:
    question_normalisee = normaliser(question)
    sparql = generer_sparql(question_normalisee, schema)
    try:
        vars_, rows = executer_sparql(g, sparql)
        return {"sparql": sparql, "vars": vars_, "rows": rows,
                "repare": False, "erreur": None}
    except Exception as e:
        erreur = str(e)
        sparql_repare = reparer_sparql(schema, question_normalisee, sparql, erreur)
        try:
            vars_, rows = executer_sparql(g, sparql_repare)
            return {"sparql": sparql_repare, "vars": vars_, "rows": rows,
                    "repare": True, "erreur": None}
        except Exception as e2:
            return {"sparql": sparql_repare, "vars": [], "rows": [],
                    "repare": True, "erreur": str(e2)}

In [11]:
INSTRUCTIONS_REPARATION = """
La requete SPARQL precedente a echoue. Corrige-la.
Utilise uniquement les URIs completes entre < >.
Retourne SEULEMENT le bloc ```sparql sans aucun texte avant.
"""

### 6. Evaluation sur 5 questions

On compare la reponse baseline (LLM seul) vs RAG (LLM + KB) sur 5 questions.
Ce tableau sera integre dans le rapport final.

In [12]:
QUESTIONS_EVAL = [
    "Quel est le genre musical de Daft Punk ?",
    "Quel label discographique a signe Radiohead ?",
    "Quels sont les membres de Pink Floyd ?",
    "Quels artistes ont le genre electronic dans la KB ?",
    "Quel artiste a influence Nirvana ?",
]

resultats_eval = []

for i, question in enumerate(QUESTIONS_EVAL, 1):
    print(f"{'='*60}")
    print(f"Question {i} : {question}")
    print()

    # Baseline
    print("--- BASELINE (sans KB) ---")
    baseline = reponse_baseline(question)
    print(baseline[:300])
    print()

    # RAG
    print("--- RAG (avec KB) ---")
    resultat = repondre_avec_rag(g, schema, question)
    afficher_resultat(question, resultat)

    resultats_eval.append({
        "question":   question,
        "baseline":   baseline[:200],
        "rag_rows":   len(resultat["rows"]),
        "rag_erreur": resultat["erreur"],
        "repare":     resultat["repare"]
    })

print("Evaluation terminee.")

Question 1 : Quel est le genre musical de Daft Punk ?

--- BASELINE (sans KB) ---
Le genre musical de Daft Punk se caractérise par une combinaison d'éléments de plusieurs styles, notamment :

* Le house et la techno électronique
* La synthpop et l'électro pop
* L'electro rock et le disco
* La musique expérimentale et l'avant-garde

Cependant, Daft Punk est également connu pour so

--- RAG (avec KB) ---
Question : Quel est le genre musical de Daft Punk ?

[Aucun resultat retourne]
[Repare automatiquement : False]

Question 2 : Quel label discographique a signe Radiohead ?

--- BASELINE (sans KB) ---
Le label discographique qui a signé Radiohead est EMI Records, plus précisément l'unité EMI Records et Capitol Records.

--- RAG (avec KB) ---
Question : Quel label discographique a signe Radiohead ?

[Resultats]
label | labelLabel
Label Parlophone | Parlophone
[Repare automatiquement : False]

Question 3 : Quels sont les membres de Pink Floyd ?

--- BASELINE (sans KB) ---
Les membres origin

### 7. Tableau de résultats pour le rapport

In [13]:
print("TABLEAU D'EVALUATION RAG")
print(f"{'N°':<4} {'Question':<45} {'RAG OK':>6} {'Repare':>7} {'Resultats':>10}")
print("-" * 75)

for i, r in enumerate(resultats_eval, 1):
    ok      = "OUI" if r["rag_erreur"] is None and r["rag_rows"] > 0 else "NON"
    repare  = "OUI" if r["repare"] else "NON"
    nb_rows = str(r["rag_rows"]) if r["rag_erreur"] is None else "erreur"
    print(f"{i:<4} {r['question'][:44]:<45} {ok:>6} {repare:>7} {nb_rows:>10}")

TABLEAU D'EVALUATION RAG
N°   Question                                      RAG OK  Repare  Resultats
---------------------------------------------------------------------------
1    Quel est le genre musical de Daft Punk ?         NON     NON          0
2    Quel label discographique a signe Radiohead      OUI     NON          1
3    Quels sont les membres de Pink Floyd ?           OUI     NON          5
4    Quels artistes ont le genre electronic dans      NON     OUI     erreur
5    Quel artiste a influence Nirvana ?               OUI     NON          1


### 8. Demo CLI interactive

Un chatbot en ligne de commande. Tape ta question et obtiens une reponse
depuis notre KB musicale. Tape `quit` pour quitter.

In [14]:
print("=" * 60)
print("CHATBOT KB MUSICALE - RAG avec Gemma 2B")
print("Tape 'quit' pour quitter")
print("=" * 60)

while True:
    question = input("\nQuestion : ").strip()
    if question.lower() in ["quit", "exit", "q"]:
        print("Au revoir.")
        break
    if not question:
        continue

    print()
    print("--- Baseline (LLM seul) ---")
    print(reponse_baseline(question)[:400])

    print()
    print("--- RAG (LLM + KB musicale) ---")
    resultat = repondre_avec_rag(g, schema, question)
    afficher_resultat(question, resultat)
    print("[SPARQL utilise]")
    print(resultat["sparql"])

CHATBOT KB MUSICALE - RAG avec Gemma 2B
Tape 'quit' pour quitter

Question : quel est le genre musical de celine dion?

--- Baseline (LLM seul) ---
Le genre musical de Céline Dion est varié, mais principalement il s'agit de :

* Pop : Céline Dion est connue pour ses chansons pop à l'origine, telles que "Un Dernier Comptable, Quelques Mots, Les Chemins de Ma Dame" et "C'est Pour Toi".
* Adult Contemporary (AC) : Ses chansons sont souvent caractérisées par leur mélodie douce et leur lyrisme romantique, ce qui fait partie du genre Adult Contempo

--- RAG (LLM + KB musicale) ---
Question : quel est le genre musical de celine dion?

[Resultats]
genre | genreLabel
Genre R%26B | R&B
Genre R%26B | R&B
Genre R%26B | R&B
Genre R%26B | R&B
Genre R%26B | R&B
Genre R%26B | R&B
Genre R%26B | R&B
Genre R%26B | R&B
Genre R%26B | R&B
Genre pop | pop
... (150 resultats au total)
[Repare automatiquement : False]

[SPARQL utilise]
SELECT ?genre ?genreLabel WHERE {
  ?celineDion exo:hasGenre ?genre .
  ?ge

### 9. Sauvegarde des résultats

In [15]:
with open(CHEMIN + "rag_resultats.json", "w", encoding="utf-8") as f:
    json.dump(resultats_eval, f, indent=2, ensure_ascii=False)
print("Resultats sauvegardes : rag_resultats.json")

Resultats sauvegardes : rag_resultats.json
